# PW04 · Attack Merge And Metrics

用途：在 Colab 会话中执行 PW04 quality_shard，只处理一个显式 QUALITY_SHARD_INDEX 对应的 quality shard，并产出该 shard 的 quality metrics。

边界：本阶段只关心 quality runtime 的设备与 batch，不重新生成 pair plan，也不做 finalize 汇总。

## 运行参数与基础路径说明

- FAMILY_ID：切换到你要处理的 paper workflow family 时修改。
- QUALITY_SHARD_INDEX：quality_shard notebook 显式运行哪个 shard，就在这里设置对应 index。
- FORCE_RERUN：prepare 模式下清空既有 PW04 输出；quality_shard / finalize 模式只用于覆盖本阶段已存在工件。
- QUALITY_DEVICE_OVERRIDE：可选 `auto`、`cuda`、`cpu`。
- QUALITY_LPIPS_BATCH_SIZE：`None` 表示不覆盖；正整数表示 notebook 显式覆盖 LPIPS batch size。
- QUALITY_CLIP_BATCH_SIZE：`None` 表示不覆盖；正整数表示 notebook 显式覆盖 CLIP batch size。
- QUALITY_PSNR_SSIM_BATCH_SIZE：`None` 表示不覆盖；正整数表示 notebook 显式覆盖 PSNR / SSIM batch size。
- QUALITY_PSNR_SSIM_BATCH_ELEMENT_BUDGET：`None` 表示不覆盖；正整数表示 notebook 显式覆盖 PSNR / SSIM batch element budget。
- REPO_BRANCH：如果你明确要切换 notebook 使用的仓库分支，再修改。
- DRIVE_PROJECT_ROOT、REPO_ROOT：仅在你的运行目录布局与默认 Colab 布局不一致时修改。

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

NOTEBOOK_NAME = "PW04_Attack_Merge_And_Metrics"
REPO_URL = "https://github.com/RICHAAARC/CEG-WM.git"
REPO_BRANCH = "organize"
DRIVE_MOUNT_ROOT = Path("/content/drive")
LOCAL_RUNTIME_ENABLED = True
LOCAL_PROJECT_ROOT = Path("/content/CEG_WM_PaperWorkflow")
PERSISTENT_DRIVE_PROJECT_ROOT = DRIVE_MOUNT_ROOT / "MyDrive" / "CEG_WM_PaperWorkflow"
DRIVE_BUNDLE_ROOT = DRIVE_MOUNT_ROOT / "MyDrive" / "CEG_WM_PaperWorkflow_Bundles"
if LOCAL_RUNTIME_ENABLED:
    DRIVE_PROJECT_ROOT = LOCAL_PROJECT_ROOT
else:
    DRIVE_PROJECT_ROOT = PERSISTENT_DRIVE_PROJECT_ROOT
REPO_ROOT = Path("/content/ceg_wm_workspace")
SCRIPT_PATH = REPO_ROOT / "paper_workflow" / "scripts" / "PW04_Attack_Merge_And_Metrics.py"
LOCAL_HF_HOME = REPO_ROOT / "huggingface_cache"
LOCAL_HF_HUB_CACHE = LOCAL_HF_HOME / "hub"
LOCAL_TRANSFORMERS_CACHE = LOCAL_HF_HOME / "transformers"

FAMILY_ID = "paper_eval_family_geometry_rescue_v1"
QUALITY_SHARD_INDEX = 0
FORCE_RERUN = False
QUALITY_DEVICE_OVERRIDE = "auto"
QUALITY_LPIPS_BATCH_SIZE = 256
QUALITY_CLIP_BATCH_SIZE = 400
QUALITY_PSNR_SSIM_BATCH_SIZE = None
QUALITY_PSNR_SSIM_BATCH_ELEMENT_BUDGET = None

def run_checked(command, cwd=None):
    print("$", " ".join(str(part) for part in command))
    subprocess.run(command, cwd=cwd, check=True)

def print_json(title, payload):
    print(f"\n[{title}]")
    print(json.dumps(payload, ensure_ascii=False, indent=2, sort_keys=True))

## 仓库与 Local Runtime 准备

本单元只负责挂载 Drive、同步仓库、准备 local runtime、安装 notebook 依赖，并设置 Hugging Face cache 环境变量。

In [ ]:
try:
    from google.colab import drive  # type: ignore
    drive.mount(str(DRIVE_MOUNT_ROOT), force_remount=True)
    DRIVE_MOUNT_STATUS = "mounted"
except Exception as exc:
    DRIVE_MOUNT_STATUS = f"skipped: {type(exc).__name__}: {exc}"

DRIVE_PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_BUNDLE_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_HF_HOME.mkdir(parents=True, exist_ok=True)
LOCAL_HF_HUB_CACHE.mkdir(parents=True, exist_ok=True)
LOCAL_TRANSFORMERS_CACHE.mkdir(parents=True, exist_ok=True)

if REPO_ROOT.exists() and (REPO_ROOT / ".git").exists():
    origin_result = subprocess.run(
        ["git", "-C", str(REPO_ROOT), "remote", "get-url", "origin"],
        check=False,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    origin_url = origin_result.stdout.strip()
    if origin_url.rstrip("/") != REPO_URL.rstrip("/"):
        shutil.rmtree(REPO_ROOT)
        run_checked(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])
    else:
        run_checked(["git", "fetch", "origin"], cwd=REPO_ROOT)
        run_checked(["git", "checkout", REPO_BRANCH], cwd=REPO_ROOT)
        run_checked(["git", "pull", "origin", REPO_BRANCH], cwd=REPO_ROOT)
else:
    if REPO_ROOT.exists():
        shutil.rmtree(REPO_ROOT)
    run_checked(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from paper_workflow.scripts.pw_local_runtime import prepare_local_runtime_for_stage
from scripts.notebook_runtime_common import resolve_notebook_model_cache_layout

if LOCAL_RUNTIME_ENABLED:
    prepare_local_runtime_for_stage(
        stage_name="PW04_Attack_Merge_And_Metrics_quality_shard",
        family_id=FAMILY_ID,
        local_project_root=LOCAL_PROJECT_ROOT,
        drive_bundle_root=DRIVE_BUNDLE_ROOT,
        shard_index=QUALITY_SHARD_INDEX,
        clean_before_run=True,
        include_optional_control_negative=False,
    )

MODEL_CACHE_LAYOUT = resolve_notebook_model_cache_layout(DRIVE_MOUNT_ROOT, REPO_ROOT, create_directories=True)
LOCAL_HF_HOME = MODEL_CACHE_LAYOUT["local_hf_home"]
LOCAL_HF_HUB_CACHE = MODEL_CACHE_LAYOUT["local_hf_hub_cache"]
LOCAL_TRANSFORMERS_CACHE = MODEL_CACHE_LAYOUT["local_transformers_cache"]
LOCAL_HF_HOME.mkdir(parents=True, exist_ok=True)
LOCAL_HF_HUB_CACHE.mkdir(parents=True, exist_ok=True)
LOCAL_TRANSFORMERS_CACHE.mkdir(parents=True, exist_ok=True)

run_checked([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"], cwd=REPO_ROOT)
if (REPO_ROOT / "pyproject.toml").exists():
    run_checked([sys.executable, "-m", "pip", "install", "-e", str(REPO_ROOT)], cwd=REPO_ROOT)
else:
    run_checked([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")], cwd=REPO_ROOT)
run_checked([sys.executable, "-m", "pip", "install", "lpips", "open_clip_torch"], cwd=REPO_ROOT)

os.environ["HF_HOME"] = str(LOCAL_HF_HOME)
os.environ["HF_HUB_CACHE"] = str(LOCAL_HF_HUB_CACHE)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(LOCAL_HF_HUB_CACHE)
os.environ["TRANSFORMERS_CACHE"] = str(LOCAL_TRANSFORMERS_CACHE)

print_json(
    "repo_bootstrap",
    {
        "drive_mount_status": DRIVE_MOUNT_STATUS,
        "repo_url": REPO_URL,
        "repo_branch": REPO_BRANCH,
        "repo_root": str(REPO_ROOT),
        "script_path": str(SCRIPT_PATH),
        "local_hf_home": str(LOCAL_HF_HOME),
        "local_hf_hub_cache": str(LOCAL_HF_HUB_CACHE),
        "local_transformers_cache": str(LOCAL_TRANSFORMERS_CACHE),
    },
)

## 运行前 Precheck

本单元保留 notebook 可见的路径定义与 mode 相关门禁，只检查当前阶段继续执行所需的前置条件。

In [ ]:
PW04_MODE = "quality_shard"
ENABLE_TAIL_ESTIMATION = False

FAMILY_ROOT = DRIVE_PROJECT_ROOT / "paper_workflow" / "families" / FAMILY_ID
PW02_SUMMARY_PATH = FAMILY_ROOT / "runtime_state" / "pw02_summary.json"
FINALIZE_MANIFEST_PATH = FAMILY_ROOT / "exports" / "pw02" / "paper_source_finalize_manifest.json"
CONTENT_THRESHOLD_EXPORT_PATH = FAMILY_ROOT / "exports" / "pw02" / "thresholds" / "content" / "thresholds.json"
ATTESTATION_THRESHOLD_EXPORT_PATH = FAMILY_ROOT / "exports" / "pw02" / "thresholds" / "attestation" / "thresholds.json"
ATTACK_SHARD_PLAN_PATH = FAMILY_ROOT / "manifests" / "attack_shard_plan.json"
ATTACK_EVENT_GRID_PATH = FAMILY_ROOT / "manifests" / "attack_event_grid.jsonl"
PREPARE_MANIFEST_PATH = FAMILY_ROOT / "exports" / "pw04" / "manifests" / "pw04_prepare_manifest.json"
QUALITY_ROOT = FAMILY_ROOT / "exports" / "pw04" / "quality"
QUALITY_PAIR_PLAN_PATH = QUALITY_ROOT / "quality_pair_plan.json"
SELECTED_QUALITY_SHARD_PATH = QUALITY_ROOT / "shards" / f"quality_shard_{QUALITY_SHARD_INDEX:04d}.json"
QUALITY_FINALIZE_MANIFEST_PATH = QUALITY_ROOT / "quality_finalize_manifest.json"
PW04_SUMMARY_PATH = FAMILY_ROOT / "runtime_state" / "pw04_summary.json"
PROJECT_ROOT_PRECHECK_LABEL = "项目运行根目录存在" if LOCAL_RUNTIME_ENABLED else "Drive 项目根目录存在"

PRECHECK_RESULTS = []

def record_precheck(name, ok, detail):
    PRECHECK_RESULTS.append({
        "name": str(name),
        "ok": bool(ok),
        "detail": str(detail),
    })

record_precheck("PW04_MODE 合法", PW04_MODE in {"prepare", "quality_shard", "finalize"}, PW04_MODE)
record_precheck(
    "QUALITY_SHARD_INDEX 合法（worker 模式）",
    PW04_MODE != "quality_shard" or (isinstance(QUALITY_SHARD_INDEX, int) and QUALITY_SHARD_INDEX >= 0),
    str(QUALITY_SHARD_INDEX),
)

ATTACK_SHARD_PLAN = {}
EXPECTED_ATTACK_EVENT_COUNT = None
DISCOVERED_ATTACK_EVENT_COUNT = 0
PLANNED_SHARD_RESULTS = []
PREPARE_MANIFEST = {}
EXPECTED_QUALITY_SHARD_PATHS = []
if ATTACK_SHARD_PLAN_PATH.exists():
    ATTACK_SHARD_PLAN = json.loads(ATTACK_SHARD_PLAN_PATH.read_text(encoding="utf-8"))
    EXPECTED_ATTACK_EVENT_COUNT = ATTACK_SHARD_PLAN.get("attack_event_count")
    for shard_row in ATTACK_SHARD_PLAN.get("shards", []):
        shard_index = int(shard_row["attack_shard_index"])
        shard_manifest_path = FAMILY_ROOT / "attack_shards" / f"shard_{shard_index:04d}" / "shard_manifest.json"
        shard_exists = shard_manifest_path.exists()
        shard_status = "<absent>"
        shard_event_count = 0
        if shard_exists:
            shard_manifest = json.loads(shard_manifest_path.read_text(encoding="utf-8"))
            shard_status = str(shard_manifest.get("status", "<absent>"))
            if isinstance(shard_manifest.get("event_count"), int):
                shard_event_count = int(shard_manifest["event_count"])
            else:
                shard_event_count = len(shard_manifest.get("events", []))
            if shard_status == "completed":
                DISCOVERED_ATTACK_EVENT_COUNT += shard_event_count
        PLANNED_SHARD_RESULTS.append({
            "attack_shard_index": shard_index,
            "path": str(shard_manifest_path),
            "exists": shard_exists,
            "status": shard_status,
            "event_count": shard_event_count,
        })

if PREPARE_MANIFEST_PATH.exists():
    PREPARE_MANIFEST = json.loads(PREPARE_MANIFEST_PATH.read_text(encoding="utf-8"))
    EXPECTED_QUALITY_SHARD_PATHS = [
        Path(str(path_value))
        for path_value in PREPARE_MANIFEST.get("expected_quality_shard_paths", [])
    ]

if PW04_MODE == "prepare":
    record_precheck(PROJECT_ROOT_PRECHECK_LABEL, DRIVE_PROJECT_ROOT.exists(), str(DRIVE_PROJECT_ROOT))
    record_precheck("仓库目录存在", REPO_ROOT.exists(), str(REPO_ROOT))
    record_precheck("PW04 脚本存在", SCRIPT_PATH.exists(), str(SCRIPT_PATH))
    record_precheck("PW02_SUMMARY_PATH 存在", PW02_SUMMARY_PATH.exists(), str(PW02_SUMMARY_PATH))
    record_precheck("FINALIZE_MANIFEST_PATH 存在", FINALIZE_MANIFEST_PATH.exists(), str(FINALIZE_MANIFEST_PATH))
    record_precheck("content thresholds export 存在", CONTENT_THRESHOLD_EXPORT_PATH.exists(), str(CONTENT_THRESHOLD_EXPORT_PATH))
    record_precheck("attestation thresholds export 存在", ATTESTATION_THRESHOLD_EXPORT_PATH.exists(), str(ATTESTATION_THRESHOLD_EXPORT_PATH))
    record_precheck("ATTACK_SHARD_PLAN_PATH 存在", ATTACK_SHARD_PLAN_PATH.exists(), str(ATTACK_SHARD_PLAN_PATH))
    record_precheck("ATTACK_EVENT_GRID_PATH 存在", ATTACK_EVENT_GRID_PATH.exists(), str(ATTACK_EVENT_GRID_PATH))
    record_precheck(
        "所有计划内 PW03 shard manifest 存在且 completed",
        all(item["exists"] and item["status"] == "completed" for item in PLANNED_SHARD_RESULTS),
        json.dumps(PLANNED_SHARD_RESULTS, ensure_ascii=False, indent=2),
    )
    record_precheck(
        "expected_attack_event_count == discovered_attack_event_count",
        isinstance(EXPECTED_ATTACK_EVENT_COUNT, int) and EXPECTED_ATTACK_EVENT_COUNT == DISCOVERED_ATTACK_EVENT_COUNT,
        f"expected_attack_event_count={EXPECTED_ATTACK_EVENT_COUNT}, discovered_attack_event_count={DISCOVERED_ATTACK_EVENT_COUNT}",
    )
elif PW04_MODE == "quality_shard":
    record_precheck("PREPARE_MANIFEST_PATH 存在", PREPARE_MANIFEST_PATH.exists(), str(PREPARE_MANIFEST_PATH))
    record_precheck("QUALITY_PAIR_PLAN_PATH 存在", QUALITY_PAIR_PLAN_PATH.exists(), str(QUALITY_PAIR_PLAN_PATH))
    record_precheck("PW04 summary 尚未关闭（worker 模式）", not PW04_SUMMARY_PATH.exists(), str(PW04_SUMMARY_PATH))
    record_precheck(
        "prepare manifest 已声明当前 quality shard",
        any(Path(str(path_obj)).name == SELECTED_QUALITY_SHARD_PATH.name for path_obj in EXPECTED_QUALITY_SHARD_PATHS),
        json.dumps([str(path_obj) for path_obj in EXPECTED_QUALITY_SHARD_PATHS], ensure_ascii=False, indent=2),
    )
else:
    record_precheck("PREPARE_MANIFEST_PATH 存在", PREPARE_MANIFEST_PATH.exists(), str(PREPARE_MANIFEST_PATH))
    record_precheck("QUALITY_PAIR_PLAN_PATH 存在", QUALITY_PAIR_PLAN_PATH.exists(), str(QUALITY_PAIR_PLAN_PATH))
    record_precheck(
        "prepare manifest 已声明 quality shards",
        bool(EXPECTED_QUALITY_SHARD_PATHS),
        json.dumps([str(path_obj) for path_obj in EXPECTED_QUALITY_SHARD_PATHS], ensure_ascii=False, indent=2),
    )
    record_precheck(
        "全部计划内 quality shard 已完成",
        bool(EXPECTED_QUALITY_SHARD_PATHS) and all(path_obj.exists() for path_obj in EXPECTED_QUALITY_SHARD_PATHS),
        json.dumps([str(path_obj) for path_obj in EXPECTED_QUALITY_SHARD_PATHS], ensure_ascii=False, indent=2),
    )
    record_precheck(
        "PW04_SUMMARY_PATH 尚未存在或允许覆盖",
        FORCE_RERUN or not PW04_SUMMARY_PATH.exists(),
        str(PW04_SUMMARY_PATH),
    )

PW04_PRECHECK_CONTEXT = {
    "pw04_mode": PW04_MODE,
    "quality_shard_index": QUALITY_SHARD_INDEX,
    "prepare_manifest_path": str(PREPARE_MANIFEST_PATH),
    "quality_pair_plan_path": str(QUALITY_PAIR_PLAN_PATH),
    "selected_quality_shard_path": str(SELECTED_QUALITY_SHARD_PATH),
    "quality_finalize_manifest_path": str(QUALITY_FINALIZE_MANIFEST_PATH),
    "pw04_summary_path": str(PW04_SUMMARY_PATH),
    "expected_quality_shard_paths": [str(path_obj) for path_obj in EXPECTED_QUALITY_SHARD_PATHS],
}

print_json("pw04_precheck", {"items": PRECHECK_RESULTS, "context": PW04_PRECHECK_CONTEXT})
hard_fail = [item for item in PRECHECK_RESULTS if not item["ok"]]
if hard_fail:
    raise RuntimeError(f"PW04 precheck failed: {hard_fail}")

## Quality Runtime 配置说明

- QUALITY_DEVICE_OVERRIDE：可选 auto、cuda、cpu。默认建议保持 auto。
- QUALITY_LPIPS_BATCH_SIZE、QUALITY_CLIP_BATCH_SIZE、QUALITY_PSNR_SSIM_BATCH_SIZE 与 QUALITY_PSNR_SSIM_BATCH_ELEMENT_BUDGET 可以直接在第一个代码 cell 设置。
- None 表示不覆盖，继续使用 notebook/runtime 默认解析结果。
- notebook override 优先于环境变量 PW_QUALITY_LPIPS_BATCH_SIZE、PW_QUALITY_CLIP_BATCH_SIZE、PW_QUALITY_PSNR_SSIM_BATCH_SIZE 与 PW_QUALITY_PSNR_SSIM_BATCH_ELEMENT_BUDGET。
- 默认 batch 只保留两类：CPU 使用 DEFAULT_QUALITY_BATCH_SIZE；CUDA 使用 LPIPS=128、CLIP=256。

In [ ]:
from paper_workflow.scripts.pw04_notebook_common import resolve_pw04_quality_runtime_summary

QUALITY_RUNTIME_SUMMARY = resolve_pw04_quality_runtime_summary(
    quality_device_override=QUALITY_DEVICE_OVERRIDE,
    quality_lpips_batch_size_override=QUALITY_LPIPS_BATCH_SIZE,
    quality_clip_batch_size_override=QUALITY_CLIP_BATCH_SIZE,
    quality_psnr_ssim_batch_size_override=QUALITY_PSNR_SSIM_BATCH_SIZE,
    quality_psnr_ssim_batch_element_budget_override=QUALITY_PSNR_SSIM_BATCH_ELEMENT_BUDGET,
    base_env=os.environ,
)
print_json("pw04_quality_runtime", QUALITY_RUNTIME_SUMMARY)

## PW04 主执行

In [ ]:
from datetime import datetime, timezone
import time

from paper_workflow.scripts.pw04_notebook_common import (
    build_gpu_peak_notebook_summary,
    build_pw04_command,
    build_pw04_subprocess_env,
    load_gpu_peak_summary,
    resolve_pw04_expected_output,
)
from paper_workflow.scripts.pw_local_runtime import archive_local_runtime_for_stage
from scripts.notebook_runtime_common import (
    build_stage_runtime_diagnostics_payload,
    build_stage_runtime_workload_summary,
    write_stage_runtime_diagnostics,
)

try:
    from scripts.gpu_session_peak import build_gpu_peak_display_summary as _build_gpu_peak_display_summary
except Exception as exc:
    GPU_PEAK_DISPLAY_HELPER = None
    GPU_PEAK_DISPLAY_HELPER_IMPORT_ERROR = f"{type(exc).__name__}: {exc}"
else:
    GPU_PEAK_DISPLAY_HELPER = _build_gpu_peak_display_summary
    GPU_PEAK_DISPLAY_HELPER_IMPORT_ERROR = None

PW04_COMMAND_KWARGS = {
    "script_path": SCRIPT_PATH,
    "drive_project_root": DRIVE_PROJECT_ROOT,
    "family_id": FAMILY_ID,
    "pw04_mode": PW04_MODE,
    "quality_shard_index": QUALITY_SHARD_INDEX,
    "force_rerun": FORCE_RERUN,
    "enable_tail_estimation": ENABLE_TAIL_ESTIMATION,
}
if PW04_MODE == "prepare":
    PW04_COMMAND_KWARGS["quality_shard_count"] = QUALITY_SHARD_COUNT
COMMAND = build_pw04_command(**PW04_COMMAND_KWARGS)
EXPECTED_OUTPUT_LABEL, EXPECTED_OUTPUT_PATH = resolve_pw04_expected_output(
    pw04_mode=PW04_MODE,
    prepare_manifest_path=PREPARE_MANIFEST_PATH,
    selected_quality_shard_path=SELECTED_QUALITY_SHARD_PATH,
    pw04_summary_path=PW04_SUMMARY_PATH,
)
PW04_SUBPROCESS_ENV = build_pw04_subprocess_env(
    repo_root=REPO_ROOT,
    base_env=os.environ,
    quality_runtime_summary=QUALITY_RUNTIME_SUMMARY,
    pw04_mode=PW04_MODE,
)
PW04_SUBPROCESS_QUALITY_RUNTIME = {
    "requested_device": QUALITY_RUNTIME_SUMMARY.get("requested_device", "<absent>"),
    "requested_device_source": QUALITY_RUNTIME_SUMMARY.get("requested_device_source", "<absent>"),
    "selected_device": PW04_SUBPROCESS_ENV.get("PW_QUALITY_TORCH_DEVICE", "<absent>"),
    "lpips_batch_size": PW04_SUBPROCESS_ENV.get("PW_QUALITY_LPIPS_BATCH_SIZE", "<absent>"),
    "lpips_batch_size_source": QUALITY_RUNTIME_SUMMARY.get("lpips_batch_size_source", "<absent>"),
    "clip_batch_size": PW04_SUBPROCESS_ENV.get("PW_QUALITY_CLIP_BATCH_SIZE", "<absent>"),
    "clip_batch_size_source": QUALITY_RUNTIME_SUMMARY.get("clip_batch_size_source", "<absent>"),
    "batch_default_reason": QUALITY_RUNTIME_SUMMARY.get("batch_default_reason", "<absent>"),
    "selection_reason": QUALITY_RUNTIME_SUMMARY.get("selection_reason", "<absent>"),
    "fallback_reason": QUALITY_RUNTIME_SUMMARY.get("fallback_reason", "<absent>"),
    "warnings": QUALITY_RUNTIME_SUMMARY.get("warnings", []),
}
print_json("pw04_subprocess_quality_runtime", PW04_SUBPROCESS_QUALITY_RUNTIME)

GPU_PEAK_SCRIPT_PATH = REPO_ROOT / "scripts" / "gpu_session_peak.py"
if PW04_MODE == "quality_shard":
    GPU_PEAK_PATH_STEM = f"pw04_quality_shard_{QUALITY_SHARD_INDEX:04d}"
else:
    GPU_PEAK_PATH_STEM = f"pw04_{PW04_MODE}"
GPU_PEAK_SUMMARY_PATH = FAMILY_ROOT / "runtime_state" / f"{GPU_PEAK_PATH_STEM}_gpu_session_peak.json"
GPU_PEAK_SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
if GPU_PEAK_SUMMARY_PATH.exists():
    GPU_PEAK_SUMMARY_PATH.unlink()
if PW04_MODE == "quality_shard":
    PW04_RUNTIME_DIAGNOSTICS_PATH = (
        FAMILY_ROOT / "runtime_state" / f"pw04_quality_shard_{QUALITY_SHARD_INDEX:04d}_runtime_diagnostics.json"
    )
elif PW04_MODE == "prepare":
    PW04_RUNTIME_DIAGNOSTICS_PATH = FAMILY_ROOT / "runtime_state" / "pw04_prepare_runtime_diagnostics.json"
else:
    PW04_RUNTIME_DIAGNOSTICS_PATH = FAMILY_ROOT / "runtime_state" / "pw04_finalize_runtime_diagnostics.json"
PW04_ARCHIVE_STAGE_NAME = f"PW04_Attack_Merge_And_Metrics_{PW04_MODE}"

MONITORED_COMMAND = [
    sys.executable,
    str(GPU_PEAK_SCRIPT_PATH),
    "--output-json",
    str(GPU_PEAK_SUMMARY_PATH),
    "--label",
    NOTEBOOK_NAME,
    "--sample-interval-ms",
    "200",
    "--",
    *COMMAND,
]
PW04_GPU_MONITOR_SETUP = {
    "gpu_peak_script_path": str(GPU_PEAK_SCRIPT_PATH),
    "gpu_peak_summary_path": str(GPU_PEAK_SUMMARY_PATH),
    "gpu_peak_script_exists": GPU_PEAK_SCRIPT_PATH.exists(),
    "gpu_peak_display_helper_import_error": GPU_PEAK_DISPLAY_HELPER_IMPORT_ERROR,
    "monitored_command": MONITORED_COMMAND,
    "expected_output_label": EXPECTED_OUTPUT_LABEL,
    "expected_output_path": str(EXPECTED_OUTPUT_PATH),
}
print_json("pw04_gpu_monitor_setup", PW04_GPU_MONITOR_SETUP)

PW04_MONITOR_RAW_SUMMARY = None
PW04_MONITOR_LOAD_ERROR = None
PW04_MONITOR_FALLBACK_REASON = None
PW04_MONITOR_EXECUTION_MODE = "wrapped_command"
RUN_STARTED_AT_UTC = datetime.now(timezone.utc).isoformat()
RUN_STARTED_AT_MONOTONIC = time.perf_counter()

if not GPU_PEAK_SCRIPT_PATH.exists():
    PW04_MONITOR_EXECUTION_MODE = "direct_command_script_missing"
    PW04_MONITOR_FALLBACK_REASON = f"gpu_session_peak.py missing: {GPU_PEAK_SCRIPT_PATH}"
    PW04_RESULT = subprocess.run(
        COMMAND,
        cwd=REPO_ROOT,
        env=PW04_SUBPROCESS_ENV,
        check=False,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
else:
    PW04_RESULT = subprocess.run(
        MONITORED_COMMAND,
        cwd=REPO_ROOT,
        env=PW04_SUBPROCESS_ENV,
        check=False,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    PW04_MONITOR_RAW_SUMMARY, PW04_MONITOR_LOAD_ERROR = load_gpu_peak_summary(GPU_PEAK_SUMMARY_PATH)
    wrapped_command_started = (
        isinstance(PW04_MONITOR_RAW_SUMMARY, dict)
        and PW04_MONITOR_RAW_SUMMARY.get("wrapped_return_code") is not None
    )
    if PW04_RESULT.returncode != 0 and not wrapped_command_started:
        PW04_MONITOR_EXECUTION_MODE = "direct_command_after_wrapper_failure"
        PW04_MONITOR_FALLBACK_REASON = (
            "gpu_session_peak wrapper failed before the wrapped PW04 command produced a return code; rerun bare PW04 command"
        )
        PW04_RESULT = subprocess.run(
            COMMAND,
            cwd=REPO_ROOT,
            env=PW04_SUBPROCESS_ENV,
            check=False,
            capture_output=True,
            text=True,
            encoding="utf-8",
            errors="replace",
        )
RUN_FINISHED_AT_UTC = datetime.now(timezone.utc).isoformat()
RUN_ELAPSED_SECONDS = time.perf_counter() - RUN_STARTED_AT_MONOTONIC

GPU_PEAK_NOTEBOOK_SUMMARY = build_gpu_peak_notebook_summary(
    raw_summary=PW04_MONITOR_RAW_SUMMARY,
    monitor_status=(
        str(PW04_MONITOR_RAW_SUMMARY.get("status"))
        if isinstance(PW04_MONITOR_RAW_SUMMARY, dict) and isinstance(PW04_MONITOR_RAW_SUMMARY.get("status"), str)
        else ("skipped" if PW04_MONITOR_EXECUTION_MODE.startswith("direct_command") else "unavailable")
    ),
    monitor_error=PW04_MONITOR_LOAD_ERROR or GPU_PEAK_DISPLAY_HELPER_IMPORT_ERROR or PW04_MONITOR_FALLBACK_REASON,
    fallback_reason=PW04_MONITOR_FALLBACK_REASON,
    display_helper=GPU_PEAK_DISPLAY_HELPER,
    gpu_peak_summary_path=GPU_PEAK_SUMMARY_PATH,
)
GPU_PEAK_NOTEBOOK_SUMMARY["monitor_execution_mode"] = PW04_MONITOR_EXECUTION_MODE
print_json("gpu_session_peak_summary", GPU_PEAK_NOTEBOOK_SUMMARY)

if PW04_RESULT.returncode != 0:
    raise RuntimeError(
        "PW04 failed: "
        f"return_code={PW04_RESULT.returncode} "
        f"monitor_status={GPU_PEAK_NOTEBOOK_SUMMARY.get('monitor_status')} "
        f"monitor_execution_mode={PW04_MONITOR_EXECUTION_MODE} "
        f"stdout={PW04_RESULT.stdout} stderr={PW04_RESULT.stderr}"
    )
if not EXPECTED_OUTPUT_PATH.exists():
    raise FileNotFoundError(
        f"PW04 {EXPECTED_OUTPUT_LABEL} file missing after successful execution: {EXPECTED_OUTPUT_PATH}"
    )

PW04_MODE_OUTPUT = json.loads(EXPECTED_OUTPUT_PATH.read_text(encoding="utf-8"))
PW04_QUALITY_PAIR_PLAN = {}
if QUALITY_PAIR_PLAN_PATH.exists():
    PW04_QUALITY_PAIR_PLAN = json.loads(QUALITY_PAIR_PLAN_PATH.read_text(encoding="utf-8"))
if PW04_MODE == "finalize":
    PW04_SUMMARY = dict(PW04_MODE_OUTPUT)
else:
    PW04_SUMMARY = None
print_json("pw04_mode_output", PW04_MODE_OUTPUT)

def runtime_count(value):
    return int(value) if isinstance(value, int) and not isinstance(value, bool) and value >= 0 else 0

PW04_QUALITY_SHARD_COUNT = runtime_count(PW04_QUALITY_PAIR_PLAN.get("quality_shard_count"))
if PW04_QUALITY_SHARD_COUNT == 0:
    PW04_QUALITY_SHARD_COUNT = runtime_count(PW04_MODE_OUTPUT.get("quality_shard_count"))
if PW04_QUALITY_SHARD_COUNT == 0:
    PW04_QUALITY_SHARD_COUNT = len(EXPECTED_QUALITY_SHARD_PATHS)
PW04_CLEAN_PAIR_COUNT = runtime_count(PW04_QUALITY_PAIR_PLAN.get("clean_expected_pair_count"))
PW04_ATTACK_PAIR_COUNT = runtime_count(PW04_QUALITY_PAIR_PLAN.get("attack_expected_pair_count"))
if PW04_MODE == "quality_shard":
    PW04_CLEAN_PAIR_COUNT = runtime_count(PW04_MODE_OUTPUT.get("clean_pair_count"))
    PW04_ATTACK_PAIR_COUNT = runtime_count(PW04_MODE_OUTPUT.get("attack_pair_count"))
PW04_TOTAL_PAIR_COUNT = PW04_CLEAN_PAIR_COUNT + PW04_ATTACK_PAIR_COUNT
PW04_COMPLETED_QUALITY_SHARD_COUNT = sum(1 for path_obj in EXPECTED_QUALITY_SHARD_PATHS if path_obj.exists())

if PW04_MODE == "prepare":
    PW04_COUNT_SUMMARY = {
        "expected_attack_event_count": runtime_count(PW04_MODE_OUTPUT.get("expected_attack_event_count")),
        "discovered_attack_event_count": runtime_count(PW04_MODE_OUTPUT.get("discovered_attack_event_count")),
        "completed_attack_event_count": runtime_count(PW04_MODE_OUTPUT.get("completed_attack_event_count")),
        "attacked_positive_event_count": runtime_count(PW04_MODE_OUTPUT.get("attacked_positive_event_count")),
        "attacked_negative_event_count": runtime_count(PW04_MODE_OUTPUT.get("attacked_negative_event_count")),
        "quality_shard_count": PW04_QUALITY_SHARD_COUNT,
        "clean_pair_count": PW04_CLEAN_PAIR_COUNT,
        "attack_pair_count": PW04_ATTACK_PAIR_COUNT,
        "total_pair_count": PW04_TOTAL_PAIR_COUNT,
    }
    PW04_WORKLOAD_UNIT_COUNT = PW04_COUNT_SUMMARY["completed_attack_event_count"]
    if PW04_WORKLOAD_UNIT_COUNT == 0:
        PW04_WORKLOAD_UNIT_COUNT = PW04_COUNT_SUMMARY["expected_attack_event_count"]
    PW04_WORKLOAD_SUMMARY = build_stage_runtime_workload_summary(
        unit_label="prepared_attack_events",
        unit_count=PW04_WORKLOAD_UNIT_COUNT,
        elapsed_seconds=RUN_ELAPSED_SECONDS,
    )
elif PW04_MODE == "quality_shard":
    PW04_COUNT_SUMMARY = {
        "quality_shard_count": PW04_QUALITY_SHARD_COUNT,
        "clean_pair_count": PW04_CLEAN_PAIR_COUNT,
        "attack_pair_count": PW04_ATTACK_PAIR_COUNT,
        "total_pair_count": PW04_TOTAL_PAIR_COUNT,
    }
    PW04_WORKLOAD_SUMMARY = build_stage_runtime_workload_summary(
        unit_label="quality_pairs",
        unit_count=PW04_TOTAL_PAIR_COUNT,
        elapsed_seconds=RUN_ELAPSED_SECONDS,
    )
else:
    PW04_COUNT_SUMMARY = {
        "quality_shard_count": PW04_QUALITY_SHARD_COUNT,
        "planned_quality_shard_count": len(EXPECTED_QUALITY_SHARD_PATHS),
        "completed_quality_shard_count": PW04_COMPLETED_QUALITY_SHARD_COUNT,
        "expected_attack_event_count": runtime_count(PW04_MODE_OUTPUT.get("expected_attack_event_count")),
        "completed_attack_event_count": runtime_count(PW04_MODE_OUTPUT.get("completed_attack_event_count")),
        "attacked_positive_event_count": runtime_count(PW04_MODE_OUTPUT.get("attacked_positive_event_count")),
        "attacked_negative_event_count": runtime_count(PW04_MODE_OUTPUT.get("attacked_negative_event_count")),
        "clean_pair_count": PW04_CLEAN_PAIR_COUNT,
        "attack_pair_count": PW04_ATTACK_PAIR_COUNT,
        "total_pair_count": PW04_TOTAL_PAIR_COUNT,
    }
    PW04_WORKLOAD_UNIT_COUNT = PW04_COUNT_SUMMARY["completed_quality_shard_count"]
    if PW04_WORKLOAD_UNIT_COUNT == 0:
        PW04_WORKLOAD_UNIT_COUNT = PW04_COUNT_SUMMARY["planned_quality_shard_count"]
    PW04_WORKLOAD_SUMMARY = build_stage_runtime_workload_summary(
        unit_label="quality_shards",
        unit_count=PW04_WORKLOAD_UNIT_COUNT,
        elapsed_seconds=RUN_ELAPSED_SECONDS,
    )

PW04_RUNTIME_DIAGNOSTICS = build_stage_runtime_diagnostics_payload(
    stage_name=PW04_ARCHIVE_STAGE_NAME,
    family_id=FAMILY_ID,
    expected_output_label=EXPECTED_OUTPUT_LABEL,
    expected_output_path=EXPECTED_OUTPUT_PATH,
    started_at_utc=RUN_STARTED_AT_UTC,
    finished_at_utc=RUN_FINISHED_AT_UTC,
    elapsed_seconds=RUN_ELAPSED_SECONDS,
    return_code=PW04_RESULT.returncode,
    stdout_text=PW04_RESULT.stdout,
    stderr_text=PW04_RESULT.stderr,
    count_summary=PW04_COUNT_SUMMARY,
    workload_summary=PW04_WORKLOAD_SUMMARY,
    shard_index=QUALITY_SHARD_INDEX if PW04_MODE == "quality_shard" else None,
    gpu_session_peak_path=GPU_PEAK_SUMMARY_PATH,
    monitor_status=(
        str(GPU_PEAK_NOTEBOOK_SUMMARY.get("monitor_status"))
        if isinstance(GPU_PEAK_NOTEBOOK_SUMMARY.get("monitor_status"), str)
        else None
    ),
)
write_stage_runtime_diagnostics(
    diagnostics_path=PW04_RUNTIME_DIAGNOSTICS_PATH,
    payload=PW04_RUNTIME_DIAGNOSTICS,
)
print_json("pw04_runtime_diagnostics", PW04_RUNTIME_DIAGNOSTICS)
if LOCAL_RUNTIME_ENABLED:
    archive_local_runtime_for_stage(
        stage_name=PW04_ARCHIVE_STAGE_NAME,
        family_id=FAMILY_ID,
        local_project_root=LOCAL_PROJECT_ROOT,
        drive_bundle_root=DRIVE_BUNDLE_ROOT,
        shard_index=QUALITY_SHARD_INDEX if PW04_MODE == "quality_shard" else None,
        clean_after_archive=False,
    )


## 输出读取与结果检查

In [ ]:
from paper_workflow.scripts.pw04_notebook_common import read_pw04_result_summary

PW04_RESULT_SUMMARY = read_pw04_result_summary(
    pw04_mode=PW04_MODE,
    family_root=FAMILY_ROOT,
    prepare_manifest_path=PREPARE_MANIFEST_PATH,
    selected_quality_shard_path=SELECTED_QUALITY_SHARD_PATH,
    pw04_summary_path=PW04_SUMMARY_PATH,
    gpu_peak_notebook_summary=GPU_PEAK_NOTEBOOK_SUMMARY,
)
print_json("pw04_result_summary", PW04_RESULT_SUMMARY)